# LOOP module — Trend Radar $100k paper book

This notebook loads **all available LOOP logs** from Supabase (`paper_loop_runs.log`, plus run metadata, `paper_orders`, and the open `paper_book`). There are no separate on-disk `.log` files in the repo; the nightly harness `python -m pipeline.compute.portfolio_loop` persists the last ~40 log lines per run.

**Harness:** Trend Radar breakout book on a fixed **$100,000** notional (`EQUITY = 100_000` in `portfolio_loop.py`).


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.research.loop_log_report import (
    closed_trades,
    equity_curve_from_prices,
    flatten_run_logs,
    get_client,
    load_loop_tables,
    portfolio_snapshot,
)

pd.set_option("display.max_colwidth", 120)


In [ ]:
tables = load_loop_tables()
runs_df = pd.DataFrame(tables["runs"])
orders_df = pd.DataFrame(tables["orders"])
book_df = pd.DataFrame(tables["book"])
logs_df = flatten_run_logs(tables["runs"])

print(f"Runs: {len(runs_df)} | Orders: {len(orders_df)} | Open book: {len(book_df)} | Log lines: {len(logs_df)}")
if not runs_df.empty:
    print(f"As-of range: {runs_df['asof_date'].min()} → {runs_df['asof_date'].max()}")
runs_df.tail(3)


In [ ]:
snapshot = portfolio_snapshot(tables["orders"], tables["book"])
summary = pd.Series(snapshot)
summary


In [ ]:
closed = closed_trades(tables["orders"])
closed


In [ ]:
sb = get_client()
curve = equity_curve_from_prices(sb, tables["orders"])
curve.tail()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

if not curve.empty:
    axes[0, 0].plot(curve["date"], curve["equity_usd"], marker="o")
    axes[0, 0].axhline(100_000, color="gray", ls="--", lw=1)
    axes[0, 0].set_title("Mark-to-market equity ($100k start)")
    axes[0, 0].set_ylabel("USD")
    axes[0, 0].tick_params(axis="x", rotation=45)

if not runs_df.empty:
    axes[0, 1].plot(runs_df["asof_date"], runs_df["deployed_pct"] * 100, marker="o", label="Deployed %")
    axes[0, 1].plot(runs_df["asof_date"], runs_df["open_risk_pct"] * 100, marker="s", label="Open risk %")
    axes[0, 1].set_title("Harness utilization by loop run")
    axes[0, 1].legend()
    axes[0, 1].tick_params(axis="x", rotation=45)

if not closed.empty:
    axes[1, 0].bar(closed["ticker"], closed["pnl_usd"], color=["#2ecc71" if x > 0 else "#e74c3c" for x in closed["pnl_usd"]])
    axes[1, 0].set_title("Realized PnL by closed trade")
    axes[1, 0].axhline(0, color="black", lw=0.8)

if not logs_df.empty:
    lvl = logs_df["level"].value_counts()
    axes[1, 1].bar(lvl.index.astype(str), lvl.values)
    axes[1, 1].set_title("Log message levels (all runs)")

plt.tight_layout()
out = ROOT / "pipeline" / "research" / "artifacts" / "loop_analysis_charts.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=144)
plt.show()
print("Saved", out)


In [ ]:
# Export a frozen snapshot for offline review
import json
artifact = ROOT / "pipeline" / "research" / "artifacts" / "loop_supabase_snapshot.json"
artifact.write_text(json.dumps(tables, indent=2, default=str))
print("Wrote", artifact)


## Verdict (auto-generated from latest Supabase data)

Run the cells above to refresh numbers. Interpretation guidelines:

- **Total return** near 0% with controlled open risk (~5–8%) → harness is behaving as designed.
- **Large drawdown** with repeated stop-outs → radar entries are fighting the tape; review posture filter and rank threshold.
- **Positive MTM with low realized losses** → open winners are carrying closed stop-outs (typical for trend books).

The markdown cell below prints a one-line health check after you execute the notebook.


In [ ]:
ret = snapshot["total_return_pct"]
risk = runs_df["open_risk_pct"].iloc[-1] * 100 if not runs_df.empty else None
wins = snapshot.get("win_rate_pct")

if ret >= 0:
    tone = "OK — paper book is flat/up"
elif ret > -5:
    tone = "MIXED — modest drawdown; risk caps look active"
else:
    tone = "STRESSED — drawdown beyond -5%; review entries/exits"

print(f"Health: {tone}")
print(f"MTM return: {ret:+.2f}% | Realized: ${snapshot['realized_pnl_usd']:+,.0f} | Unrealized: ${snapshot['unrealized_pnl_usd']:+,.0f}")
if risk is not None:
    print(f"Latest open risk: {risk:.1f}% of $100k | Open names: {snapshot['open_names']}")
if wins is not None:
    print(f"Closed trade win rate: {wins:.0f}%")
